# 🧠 คำอธิบายและตัวอย่างการปฏิบัติการตัวจำแนกประเภทเนอีฟเบย์แบบเกาส์เซียน (Gaussian Naive Bayes)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **Naive Bayes**! ในโน้ตบุ๊กนี้เราจะ:
1. สร้างชุดข้อมูลจำลองตามกรณีศึกษาการประเมินกล่อง Bounding Box ระหว่างวัตถุจริง (`real-object` คลาส +1) และสัญญาณลวง (`false-positive` คลาส 0)
2. ฝึกสอนแบบจำลองตัวจำแนกประเภท **Gaussian Naive Bayes** โดยใช้ไลบรารี `scikit-learn`
3. พล็อตกราฟแสดงขอบเขตการตัดสินใจที่เป็นเส้นโค้งพาราโบลา/กำลังสอง (Quadratic Decision Boundaries) ซึ่งถูกคำนวณขึ้นจากโมเดล
4. พัฒนาแบบจำลอง **Gaussian Naive Bayes จากศูนย์ (from scratch)** ด้วยภาษา Python/NumPy:
   - เรียนรู้ค่าความน่าจะเป็นตั้งต้น (Class Priors), ค่าเฉลี่ยของคุณลักษณะ ($\mu$), และค่าความแปรปรวน ($\sigma^2$) ของคุณลักษณะสำหรับแต่ละคลาส
   - คำนวณหาค่าความหนาแน่นความน่าจะเป็นแบบเกาส์เซียน (Gaussian Probability Densities)
   - คำนวณความน่าจะเป็นภายหลังแบบลอการิทึม (Log-posteriors) เพื่อป้องกันปัญหาความคลาดเคลื่อนเชิงตัวเลขเลขทศนิยมขนาดเล็ก (Numerical Underflow):
     $$\log P(C_k | \mathbf{x}) \propto \log P(C_k) - \frac{1}{2} \sum_{j=1}^{n} \left[ \log(2\pi \sigma_{k,j}^2) + \frac{(x_j - \mu_{k,j})^2}{\sigma_{k,j}^2} \right]$$
5. ประเมินผลความถูกต้องของโมเดลที่พัฒนาขึ้นเองเปรียบเทียบกับไลบรารีมาตรฐาน scikit-learn

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. การสร้างข้อมูลตามกรณีศึกษา (Case Study Data Generation)

เราจะสุ่มสร้างคุณลักษณะของกล่อง Bounding Box 100 ตัวอย่าง ประกอบด้วย 2 คุณลักษณะที่เป็นจำนวนจริงต่อเนื่อง:
1.  `ratio` อัตราส่วนความสูงต่อความกว้าง (Height-to-width ratio)
2.  `center_y` พิกัดกึ่งกลางแนวตั้งบนรูปภาพ (Vertical center pixel coordinate)

คลาสข้อมูลเป้าหมาย:
*   คลาส 1 (`real-object` วัตถุจริง): จุดศูนย์กลางของคลาสอยู่ที่ ratio 1.5, center_y 240 (อยู่บริเวณกลางหน้าจอภาพถ่าย)
*   คลาส 0 (`false-positive` สัญญาณลวง): จุดศูนย์กลางของคลาสอยู่ที่ ratio 0.8, center_y 420 (สะท้อนพฤติกรรมเงาบนพื้นถนนที่ทำให้ตัวตรวจจับโมเดลวาดกล่องข้อความเท็จขึ้นมา)

In [ ]:
m = 100

# คลาส 1: วัตถุที่แท้จริง (Real Objects)
X_real = np.random.randn(m // 2, 2) * np.array([0.25, 40]) + np.array([1.5, 240])
y_real = np.ones(m // 2)

# คลาส 0: สัญญาณลวงจากสิ่งรบกวนหรือเงา (False Positives)
X_fake = np.random.randn(m // 2, 2) * np.array([0.20, 30]) + np.array([0.8, 420])
y_fake = np.zeros(m // 2)

# รวมชุดข้อมูลฝึกสอน
X_train = np.vstack((X_real, X_fake))
y_train = np.concatenate((y_real, y_fake))

# พล็อตกราฟจุดกระจายตัวของข้อมูลคัดกรอง
plt.figure(figsize=(8, 5))
plt.scatter(X_real[:, 0], X_real[:, 1], color='blue', label='Class 1: Real Object', alpha=0.7)
plt.scatter(X_fake[:, 0], X_fake[:, 1], color='red', label='Class 0: False Positive', alpha=0.7)
plt.xlabel('Height-to-Width Ratio')
plt.ylabel('Center Y Coordinate')
plt.title('Bounding Box Verification Dataset')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 2. การสร้าง Gaussian Naive Bayes ด้วย Scikit-Learn

ลองฝึกสอนแบบจำลองด้วยคลาส `GaussianNB` ของ scikit-learn และเรียกดูค่าความน่าจะเป็นก่อนหน้า (Priors), ค่าเฉลี่ย และค่าความแปรปรวนที่แบบจำลองเรียนรู้ได้จากข้อมูลฝึกสอน

In [ ]:
# ฝึกสอนแบบจำลอง
gnb_sklearn = GaussianNB()
gnb_sklearn.fit(X_train, y_train)

# แสดงพารามิเตอร์ที่โมเดลเรียนรู้ได้
print("--- Class Parameters Learned by Scikit-Learn ---")
print("Class Priors    :", gnb_sklearn.class_prior_)
print("Class Means (μ) :\n", gnb_sklearn.theta_)
print("Class Variances (σ²):\n", gnb_sklearn.var_)

# ประเมินผลค่าความถูกต้องพยากรณ์
y_pred_sklearn = gnb_sklearn.predict(X_train)
print(f"\nTraining Accuracy: {accuracy_score(y_train, y_pred_sklearn) * 100:.2f}%")

## 3. การสร้าง Gaussian Naive Bayes จากศูนย์ด้วย NumPy (Gaussian Naive Bayes from Scratch)

เรามาลองเขียนโมเดลขึ้นมาเองโดยใช้ NumPy กันครับ
ในขั้นตอนการเทรน (`fit`):
1.  คำนวณค่า Prior ของแต่ละคลาส: $P(C_k) = \frac{m_k}{m}$
2.  คำนวณหาค่าเฉลี่ย $\mu_{k,j}$ และความแปรปรวน $\sigma_{k,j}^2$ สำหรับแต่ละคุณลักษณะ $j$ และระดับคลาส $k$

ในขั้นตอนทำนายผล (`predict`):
คำนวณค่าความน่าจะเป็นหลังแบบลอการิทึม (Log Posterior Probability) สำหรับแต่ละคลาส $C_k$:
$$\log P(C_k | \mathbf{x}) \propto \log P(C_k) + \sum_{j=1}^{n} \log P(x_j | C_k)$$

โดยที่:
$$\log P(x_j | C_k) = -\frac{1}{2} \log(2\pi \sigma_{k,j}^2) - \frac{(x_j - \mu_{k,j})^2}{2\sigma_{k,j}^2}$$

และทำการเลือกทำนายคลาสที่มีค่า Log Posterior สูงที่สุดออกมา

In [ ]:
class CustomGaussianNB:
    def __init__(self):
        self.classes = None
        self.priors = {}
        self.means = {}
        self.vars = {}

    def fit(self, X, y):
        self.classes = np.unique(y)
        m = X.shape[0]
        
        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = X_c.shape[0] / m
            self.means[c] = np.mean(X_c, axis=0)
            self.vars[c] = np.var(X_c, axis=0)

    def _pdf(self, x, mean, var):
        # คำนวณฟังก์ชันความหนาแน่นความน่าจะเป็นแบบเกาส์เซียน (Gaussian PDF)
        num = np.exp(-((x - mean) ** 2) / (2 * var))
        den = np.sqrt(2 * np.pi * var)
        return num / den

    def _predict_single(self, x):
        posteriors = []
        
        for c in self.classes:
            log_prior = np.log(self.priors[c])
            eps = 1e-15
            pdfs = self._pdf(x, self.means[c], self.vars[c])
            log_likelihood = np.sum(np.log(pdfs + eps))
            
            log_posterior = log_prior + log_likelihood
            posteriors.append((log_posterior, c))
            
        return max(posteriors)[1]

    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])

# เทรนแบบจำลองจำลองที่เราสร้างขึ้นเองจากศูนย์
gnb_scratch = CustomGaussianNB()
gnb_scratch.fit(X_train, y_train)

# ประเมินค่าความแม่นยำของ Custom GNB
y_pred_scratch = gnb_scratch.predict(X_train)
scratch_acc = accuracy_score(y_train, y_pred_scratch)

print(f"Custom Scratch Gaussian NB Accuracy: {scratch_acc * 100:.2f}%")

## 4. การแสดงขอบเขตการตัดสินใจ (Decision Boundary Visualization)

เนื่องจากแบบจำลอง Gaussian Naive Bayes มีความสามารถในการคำนวณค่าเฉลี่ยและความแปรปรวนของแต่ละคลาสแยกอิสระต่อกัน จึงส่งผลให้แบบจำลองสามารถลากเส้นแบ่งเขตการตัดสินใจที่มีลักษณะโค้งเป็นกราฟกำลังสอง (Quadratic Decision Boundary) ได้อย่างน่าทึ่ง เรามาลองพล็อตกราฟดูกันครับ!

In [ ]:
from matplotlib.colors import ListedColormap

# สร้าง Grid พิกัดสำหรับแสดงขอบเขตการจำแนกคลาส
x_min, x_max = X_train[:, 0].min() - 0.2, X_train[:, 0].max() + 0.2
y_min, y_max = X_train[:, 1].min() - 30, X_train[:, 1].max() + 30
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01), 
                     np.arange(y_min, y_max, 1))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# ทำนายผลลัพธ์บนพื้นที่ Grid
Z = gnb_scratch.predict(grid_points)
Z = Z.reshape(xx.shape)

# พล็อตกราฟจำลองเขตแดนการตัดสินใจ
plt.figure(figsize=(10, 6))
cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold = ['red', 'blue']

plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.5)
plt.scatter(X_real[:, 0], X_real[:, 1], color='blue', label='Class 1: Real Object', alpha=0.6, edgecolor='k')
plt.scatter(X_fake[:, 0], X_fake[:, 1], color='red', label='Class 0: False Positive', alpha=0.6, edgecolor='k')

plt.xlabel('Height-to-Width Ratio')
plt.ylabel('Center Y Coordinate')
plt.title('Gaussian Naive Bayes Decision Boundary')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

สังเกตว่าขอบเขตการตัดสินใจที่ลากออกมาระหว่างคลาส 0 และ 1 มีลักษณะโค้งมนไม่เป็นเส้นตรง (Non-linear curved parabolic boundary) ซึ่งเหมาะอย่างยิ่งสำหรับการแบ่งคลาสของข้อมูลที่มีการทับซ้อนหรือกระจายตัวเยื้องห่างออกจากกันตามธรรมชาติของตัวเลขจำนวนจริงต่อเนื่อง!

## 💡 ความเชื่อมโยงสู่ Deep Learning และ YOLO
*   **การกำหนดค่าน้ำหนักล่วงหน้าด้วยความน่าจะเป็น (Prior Probability Initialization):** ในแบบจำลองตรวจจับวัตถุระดับลึกอย่าง YOLO ชั้นทำหน้าที่จำแนกประเภท (Classification Layers) จะได้รับการตั้งค่าการเริ่มต้นน้ำหนักด้วยค่า Prior พิเศษเฉพาะตัว เนื่องจากในเฟรมภาพถ่ายจะมีเศษสัดส่วนที่เป็นฉากหลังหรือที่ว่างปะปนอยู่มากกว่าวัตถุเป้าหมายจริงอยู่มหาศาล ค่าน้ำหนักอคติ (Bias weights) ของหัวทำนายแยกคลาสจึงเริ่มต้นการฝึกสอนด้วยสมการเทียบสัดส่วนก่อนหน้าของความน่าจะเป็น (เช่น $b = -\log((1 - p)/p)$ โดยที่ $p$ แทนค่าความน่าจะเป็นก่อนหน้าในการพบวัตถุจริง เช่น $0.01$) ซึ่งแนวคิดนี้ตรงตามทฤษฎีของเบย์ (Bayes' Theorem) ช่วยประคองทิศทางการอัปเดตน้ำหนักไม่ให้เกิดปัญหาค่าความสูญเสียพุ่งสูงจนผิดรูป (Exploding Loss) ในช่วงเริ่มต้นฝึกสอนรอบแรกๆ